# RavenStack SaaS Analytics – SQL Business Case Study

**Prepared by:** Janardhan

**Project:** SQL Business Analytics Portfolio

**Status:** Completed

---

## Project Overview

RavenStack is a fictional AI-powered collaboration platform developed as a realistic SaaS business simulation for learning SQL, Business Intelligence, and Data Analytics.

In this case study, RavenStack is preparing for its public product launch. The objective of this analysis is to investigate customer signups, subscription activity, and revenue trends to uncover insights that support business growth and decision-making.

The analysis is performed using **MySQL** for data querying and **Jupyter Notebook** for documenting the complete analytical workflow.

---

## Dataset Information

**Dataset:** RavenStack Synthetic SaaS Dataset

**Author:** River @ Rivalytics

**Tables Used:**

- accounts
- subscriptions

**Relationship:**

accounts.account_id = subscriptions.account_id

---

## Project Objectives

The objectives of this project are to:

- Understand the structure and quality of the dataset.
- Validate the integrity of relational data before analysis.
- Analyze customer and subscription behavior.
- Calculate important SaaS business metrics such as customer growth, Monthly Recurring Revenue (MRR), plan distribution, and trial-to-paid conversion.
- Solve real-world business problems using SQL.
- Build a professional SQL portfolio project.
- Prepare the dataset for an interactive Power BI dashboard.

---

## Tools Used

- MySQL
- Jupyter Notebook
- Python (Pandas + SQLAlchemy)
- Power BI

---

## Project Workflow

1. Project Introduction
2. Database Connection
3. Phase 0 – Database Familiarization
4. Business SQL Analysis (40–60 Business Problems)
5. Business Insights
6. Power BI Dashboard Development
7. Final Conclusion

In [56]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = "root"
password = quote_plus("YOUR_MYSQL_PASSWORD")   # @ must be encoded
host = "127.0.0.1"
port = "3306"
database = "ravenstack_analytics"

eng = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

df = pd.read_sql("SELECT * FROM accounts LIMIT 5;", eng)



# Phase 0 – Database Familiarization.

## Check 1 – Current Database

**Objective**

Verify that the notebook is connected to the correct MySQL database before starting any analysis.

In [57]:
query = ''' select Database() '''
pd.read_sql(query , eng)

,Database()
0,ravenstack_analytics


## Check 2 – Available Tables

### Objective

Verify that all required tables are present in the database before starting the analysis.

In [58]:
query = ''' show tables ;'''
pd.read_sql(query,eng)

,Tables_in_ravenstack_analytics
0,accounts
1,subscriptions


## Check 3 – Table Structure (accounts)

### Objective

Understand the structure of the **accounts** table by examining its column names, data types, NULL constraints, and key information. This helps verify that the table is correctly designed before performing business analysis.

In [59]:
query = """
DESCRIBE accounts;
"""

pd.read_sql(query, eng)

,Field,Type,Null,Key,Default,Extra
0,account_id,text,YES,,None,
1,account_name,text,YES,,None,
2,industry,text,YES,,None,
3,country,text,YES,,None,
4,signup_date,text,YES,,None,
5,referral_source,text,YES,,None,
6,plan_tier,text,YES,,None,
7,seats,int,YES,,None,
8,is_trial,text,YES,,None,
9,churn_flag,text,YES,,None,


### Observation

- The `accounts` table contains customer-related information.
- Most categorical columns are stored as **TEXT**.
- The `seats` column is stored as **INT**.
- The `signup_date` column is currently stored as **TEXT** instead of `DATE`.
- The `is_trial` and `churn_flag` columns are stored as **TEXT** instead of Boolean.
- Although some data types are not ideal, the table can still be used for SQL analysis. Data type conversions can be applied later if required.

## Check 4 – Table Structure (subscriptions)

### Objective

Understand the structure of the **subscriptions** table by examining its column names, data types, NULL constraints, and key information. This helps verify that the table is correctly designed and ready for subscription and revenue analysis.

In [60]:
query = """
DESCRIBE subscriptions;
"""

pd.read_sql(query, eng)

,Field,Type,Null,Key,Default,Extra
0,subscription_id,text,YES,,None,
1,account_id,text,YES,,None,
2,start_date,text,YES,,None,
3,end_date,text,YES,,None,
4,plan_tier,text,YES,,None,
5,seats,int,YES,,None,
6,mrr_amount,int,YES,,None,
7,arr_amount,int,YES,,None,
8,is_trial,text,YES,,None,
9,upgrade_flag,text,YES,,None,


### Observation

- The **subscriptions** table stores subscription lifecycle and revenue information.
- Text-based columns such as `subscription_id`, `account_id`, `plan_tier`, and `billing_frequency` are stored as **TEXT**.
- Numerical columns (`seats`, `mrr_amount`, and `arr_amount`) are correctly stored as **INT**.
- Date columns (`start_date` and `end_date`) are currently stored as **TEXT** instead of `DATE`.
- Boolean columns (`is_trial`, `upgrade_flag`, `downgrade_flag`, `churn_flag`, and `auto_renew_flag`) are stored as **TEXT** instead of Boolean.
- The table structure is suitable for SQL analysis. Data type conversions can be applied later if required for advanced date or Boolean operations.

## Check 5 – Preview Data (accounts)

### Objective

Preview the first five records of the **accounts** table to understand the actual data stored in each column. This helps verify that the data has been imported correctly and provides familiarity with the dataset before performing business analysis.

In [61]:
query = """
SELECT *
FROM accounts
LIMIT 5;
"""

pd.read_sql(query, eng)

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


### Observation

- The first five records were displayed successfully.
- Customer information appears complete and well-structured.
- The dataset includes customer details such as account ID, company name, industry, country, signup date, referral source, subscription plan, and seat count.
- The `signup_date` values are displayed in a consistent `YYYY-MM-DD` format.
- Boolean columns (`is_trial` and `churn_flag`) contain valid `True/False` values.
- No obvious data quality issues were observed in the preview.
- The `accounts` table is ready for further validation and business analysis.

## Check 6 – Preview Data (subscriptions)

### Objective

Preview the first five records of the **subscriptions** table to understand the actual subscription data, verify that the records have been imported correctly, and become familiar with the columns before performing business analysis.

In [62]:
query = """
SELECT *
FROM subscriptions
LIMIT 5;
"""

pd.read_sql(query, eng)

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,,Enterprise,27,5373,64476,False,False,False,False,monthly,True


### Observation

- The first five subscription records were displayed successfully.
- The table contains subscription details such as subscription ID, account ID, plan tier, seats, MRR, ARR, billing frequency, and subscription status.
- Blank values in the `end_date` column likely indicate active subscriptions.
- Trial subscriptions are represented with `is_trial = True`, and some trial records have zero MRR and ARR values.
- Revenue columns (`mrr_amount` and `arr_amount`) contain valid numeric values.
- Both monthly and annual billing frequencies are present.
- The data appears consistent and suitable for subscription and revenue analysis.

## Check 7 – Record Counts

### Objective

Verify the number of records in each table and compare them with the dataset documentation. This confirms that the data has been imported completely before starting business analysis.

In [63]:
query = """
SELECT COUNT(*) AS total_accounts
FROM accounts;
"""

pd.read_sql(query, eng)

,total_accounts
0,500


In [64]:
query = """
SELECT COUNT(*) AS total_subscriptions
FROM subscriptions;
"""

pd.read_sql(query, eng)

,total_subscriptions
0,5000


### Observation

- The `accounts` table contains **500** records.
- The `subscriptions` table contains **5000** records.
- The record counts match the dataset documentation, indicating that both tables were imported successfully without any missing records.

## Check 8 – Explore Unique Values

### Objective

Explore the unique values in important categorical columns to understand the available categories within the dataset. This helps identify the different customer segments, subscription plans, acquisition channels, and billing options before performing business analysis.

In [65]:
query = """
SELECT DISTINCT industry
FROM accounts;
"""

pd.read_sql(query, eng)

,industry
0,EdTech
1,FinTech
2,DevTools
3,HealthTech
4,Cybersecurity


In [66]:
query = """
SELECT DISTINCT plan_tier
FROM accounts;
"""

pd.read_sql(query, eng)

,plan_tier
0,Basic
1,Enterprise
2,Pro


In [67]:
query = """
SELECT DISTINCT referral_source
FROM accounts;
"""

pd.read_sql(query, eng)

,referral_source
0,partner
1,other
2,organic
3,event
4,ads


In [68]:
query = """
SELECT DISTINCT billing_frequency
FROM subscriptions;
"""

pd.read_sql(query, eng)

,billing_frequency
0,monthly
1,annual


### Observation

- The dataset contains **5 industries**: EdTech, FinTech, DevTools, HealthTech, and Cybersecurity.
- There are **3 subscription plan tiers**: Basic, Pro, and Enterprise.
- Customer acquisition is tracked through **5 referral sources**: organic, ads, event, partner, and other.
- The dataset supports **2 billing frequencies**: monthly and annual.
- All categorical values are valid and consistent, indicating that the dataset is ready for grouping, filtering, and segmentation during business analysis.

## Check 9 – Duplicate Record Validation

### Objective

Verify that the primary identifier columns contain unique values. Duplicate primary keys can lead to incorrect joins, inaccurate aggregations, and unreliable business insights. This check ensures data integrity before performing business analysis.

In [69]:
query = """
SELECT
    account_id,
    COUNT(*) AS duplicate_count
FROM accounts
GROUP BY account_id
HAVING COUNT(*) > 1;
"""

pd.read_sql(query, eng)

,account_id,duplicate_count


In [70]:
query = """
SELECT
    subscription_id,
    COUNT(*) AS duplicate_count
FROM subscriptions
GROUP BY subscription_id
HAVING COUNT(*) > 1;
"""

pd.read_sql(query, eng)

,subscription_id,duplicate_count


### Observation

- No duplicate `account_id` values were found in the **accounts** table.
- No duplicate `subscription_id` values were found in the **subscriptions** table.
- Both identifier columns contain unique values, indicating good data integrity.
- The dataset is suitable for reliable joins, aggregations, and business analysis.

## Check 10 – Relationship Validation

### Objective

Verify that every subscription record is linked to a valid account. This ensures referential integrity between the `accounts` and `subscriptions` tables before performing JOIN operations.

In [71]:
query = """
SELECT COUNT(*) AS orphan_records
FROM subscriptions s
LEFT JOIN accounts a
ON s.account_id = a.account_id
WHERE a.account_id IS NULL;
"""

pd.read_sql(query, eng)

,orphan_records
0,0


### Observation

- No orphan records were found in the `subscriptions` table.
- Every `account_id` in the `subscriptions` table has a matching record in the `accounts` table.
- The relationship between the two tables is valid and can be used safely for JOIN operations and business analysis.

## Check 11 – Dataset Date Range

### Objective

Identify the earliest and latest dates in the dataset to understand the overall time period covered by the customer signup and subscription data. This helps define the scope of future trend and time-based analyses.

In [72]:
query = """
SELECT
    MIN(signup_date) AS earliest_signup_date,
    MAX(signup_date) AS latest_signup_date
FROM accounts;
"""

pd.read_sql(query, eng)

,earliest_signup_date,latest_signup_date
0,2023-01-02,2024-12-31


In [73]:
query = """
SELECT
    MIN(start_date) AS earliest_subscription_date,
    MAX(start_date) AS latest_subscription_date
FROM subscriptions;
"""

pd.read_sql(query, eng)

,earliest_subscription_date,latest_subscription_date
0,2023-01-09,2024-12-31


### Observation

- The customer signup data spans from **2023-01-02** to **2024-12-31**.
- The subscription data spans from **2023-01-09** to **2024-12-31**.
- The dataset covers approximately **two years** of business activity.
- The available date range is sufficient for performing time-based analyses such as customer growth, subscription trends, and revenue analysis.

# Phase 0 Summary

### Summary

The database connection was successfully established, and all required tables were verified. The table structures, record counts, and sample data were reviewed to understand the dataset. Data quality checks confirmed that there are no duplicate records and no orphan records between the `accounts` and `subscriptions` tables, ensuring referential integrity. The dataset spans from January 2023 to December 2024, providing sufficient historical data for business analysis.

Overall, the dataset is well-structured and ready for SQL-based business analysis.

# **Business SQL Analysis**

# Level 1: Beginner (Data Validation, Data Quality, Basic Customer/Subscription Analysis)

## Question 1

### Business Question

How many total accounts and how many total subscriptions exist in the database?

### Business Objective

Establish baseline record counts before any analysis. This is the first validation step before performing business analytics.

Difficulty : Beginner

SQL Concepts Required

- SELECT
- COUNT()

In [74]:
query = """
SELECT COUNT(*) AS total_accounts
FROM accounts;
"""

print(pd.read_sql(query, eng))

   total_accounts
0             500


In [75]:
query = """
SELECT COUNT(*) AS total_subscriptions
FROM subscriptions;
"""

pd.read_sql(query, eng)

,total_subscriptions
0,5000


### Observation

- The **accounts** table contains **500** records.
- The **subscriptions** table contains **5000** records.
- The record counts match the dataset documentation, confirming that the data was imported successfully and is ready for business analysis.

## Question 2

### Business Question: 
Are there any duplicate account_id values in the accounts table, or duplicate subscription_id values in subscriptions?

### Business Objective: 
Data quality check — primary keys must be unique before any join or aggregation is trusted.

Difficulty: Beginner

SQL Concepts Required: GROUP BY, HAVING, COUNT()


In [76]:
query = """
SELECT account_id, COUNT(*) AS duplicate_count
FROM accounts
GROUP BY account_id
HAVING COUNT(*) > 1;

"""

pd.read_sql(query, eng)

,account_id,duplicate_count


In [77]:
query = """
SELECT subscription_id, COUNT(*) AS duplicate_count
FROM subscriptions
GROUP BY subscription_id
HAVING COUNT(*) > 1;
"""

pd.read_sql(query, eng)

,subscription_id,duplicate_count


### Observation

- No duplicate `account_id` values were found in the **accounts** table.
- No duplicate `subscription_id` values were found in the **subscriptions** table.
- This confirms that the primary keys are unique, ensuring reliable joins and aggregations during subsequent business analysis.

## Question 3

### Business Question

Are there any `subscriptions.account_id` values that do not exist in the `accounts` table (orphan records)?

### Business Objective

Verify referential integrity between the `accounts` and `subscriptions` tables. This check ensures that every subscription belongs to a valid customer account before performing joins for business analysis.

 Difficulty

Beginner

 SQL Concepts Required

- LEFT JOIN
- IS NULL
- COUNT()

In [78]:
query = """
SELECT 
    COUNT(*) AS orphan_records
FROM subscriptions s
LEFT JOIN accounts a
    ON s.account_id = a.account_id
WHERE a.account_id IS NULL;
"""

pd.read_sql(query, eng)

,orphan_records
0,0


### Observation

- The query returned **0 orphan records**.
- Every `account_id` in the `subscriptions` table has a matching `account_id` in the `accounts` table.
- This confirms that the relationship between the two tables is valid and that the data is ready for reliable JOIN operations in subsequent business analysis.

## Question 4

### Business Question

Which columns in the `accounts` and `subscriptions` tables contain NULL values, and how many NULL values exist in each column?

### Business Objective

Assess data completeness before building business KPIs and performing analysis. Identifying NULL values helps determine whether missing data requires cleaning or represents valid business scenarios.

Difficulty

Beginner

SQL Concepts Required

- CASE
- IS NULL
- SUM()
- Conditional Aggregation

In [79]:
query = """
SELECT 
    SUM(CASE WHEN account_id IS NULL THEN 1 ELSE 0 END) AS null_account_id,
    SUM(CASE WHEN account_name IS NULL THEN 1 ELSE 0 END) AS null_account_name,
    SUM(CASE WHEN industry IS NULL THEN 1 ELSE 0 END) AS null_industry,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS null_country,
    SUM(CASE WHEN signup_date IS NULL THEN 1 ELSE 0 END) AS null_signup_date,
    SUM(CASE WHEN referral_source IS NULL THEN 1 ELSE 0 END) AS null_referral_source,
    SUM(CASE WHEN plan_tier IS NULL THEN 1 ELSE 0 END) AS null_plan_tier,
    SUM(CASE WHEN seats IS NULL THEN 1 ELSE 0 END) AS null_seats,
    SUM(CASE WHEN is_trial IS NULL THEN 1 ELSE 0 END) AS null_is_trial,
    SUM(CASE WHEN churn_flag IS NULL THEN 1 ELSE 0 END) AS null_churn_flag
FROM accounts;
"""

pd.read_sql(query, eng)

,null_account_id,null_account_name,null_industry,null_country,null_signup_date,null_referral_source,null_plan_tier,null_seats,null_is_trial,null_churn_flag
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [80]:
query = """
SELECT 
    SUM(CASE WHEN subscription_id IS NULL THEN 1 ELSE 0 END) AS null_subscription_id,
    SUM(CASE WHEN account_id IS NULL THEN 1 ELSE 0 END) AS null_account_id,
    SUM(CASE WHEN start_date IS NULL THEN 1 ELSE 0 END) AS null_start_date,
    SUM(CASE WHEN end_date IS NULL THEN 1 ELSE 0 END) AS null_end_date,
    SUM(CASE WHEN plan_tier IS NULL THEN 1 ELSE 0 END) AS null_plan_tier,
    SUM(CASE WHEN seats IS NULL THEN 1 ELSE 0 END) AS null_seats,
    SUM(CASE WHEN mrr_amount IS NULL THEN 1 ELSE 0 END) AS null_mrr_amount,
    SUM(CASE WHEN arr_amount IS NULL THEN 1 ELSE 0 END) AS null_arr_amount,
    SUM(CASE WHEN is_trial IS NULL THEN 1 ELSE 0 END) AS null_is_trial,
    SUM(CASE WHEN upgrade_flag IS NULL THEN 1 ELSE 0 END) AS null_upgrade_flag,
    SUM(CASE WHEN downgrade_flag IS NULL THEN 1 ELSE 0 END) AS null_downgrade_flag,
    SUM(CASE WHEN churn_flag IS NULL THEN 1 ELSE 0 END) AS null_churn_flag,
    SUM(CASE WHEN billing_frequency IS NULL THEN 1 ELSE 0 END) AS null_billing_frequency,
    SUM(CASE WHEN auto_renew_flag IS NULL THEN 1 ELSE 0 END) AS null_auto_renew_flag
FROM subscriptions;
"""

pd.read_sql(query, eng)

,null_subscription_id,null_account_id,null_start_date,null_end_date,null_plan_tier,null_seats,null_mrr_amount,null_arr_amount,null_is_trial,null_upgrade_flag,null_downgrade_flag,null_churn_flag,null_billing_frequency,null_auto_renew_flag
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Observation

#### Accounts Table

- No NULL values were found in any column.
- All customer records are complete.

#### Subscriptions Table

- No NULL values were found in any column.
- The subscription dataset is complete and suitable for business analysis.

Overall, the dataset does not require NULL value handling before proceeding with KPI calculations and analytical queries.

## Question 5

### Business Question

How many accounts exist in each industry? List the industries from the highest to the lowest number of accounts.

### Business Objective

Understand the composition of the customer base across industries. This helps identify which industries contribute the largest number of customers and supports market segmentation.

Difficulty

Beginner

SQL Concepts Required

- SELECT
- GROUP BY
- COUNT()
- ORDER BY

In [81]:
query = """
SELECT industry,
       COUNT(*) AS no_of_accounts
FROM accounts
GROUP BY industry
ORDER BY no_of_accounts DESC;
"""

pd.read_sql(query, eng)

,industry,no_of_accounts
0,DevTools,113
1,FinTech,112
2,Cybersecurity,100
3,HealthTech,96
4,EdTech,79


### Observation
The dataset contains 500 customer accounts distributed across five industries.

DevTools has the highest number of customer accounts (113), followed closely by FinTech (112).

Cybersecurity and HealthTech represent medium-sized customer segments with 100 and 96 accounts, respectively.

EdTech has the lowest number of customer accounts (79) in the dataset.

The customer distribution is relatively balanced across industries, with no single industry dominating the customer base.

## Question 6

### Business Question

How many accounts were acquired through each referral source (organic, ads, event, partner, and other)?

### Business Objective

Analyze the customer acquisition channel distribution. This helps the business understand which marketing channels bring in the most customers and supports future marketing ROI analysis.

Difficulty

Beginner

 SQL Concepts Required

- SELECT
- COUNT()
- GROUP BY
- ORDER BY

In [82]:
query = """
SELECT referral_source,
       COUNT(*) AS no_of_accounts
FROM accounts
GROUP BY referral_source
ORDER BY no_of_accounts DESC;
"""

pd.read_sql(query, eng)

,referral_source,no_of_accounts
0,organic,114
1,other,103
2,ads,98
3,event,96
4,partner,89


### Observation

- The dataset contains customer accounts acquired through five referral sources.
- **Organic** is the leading acquisition channel with **114** customer accounts.
- **Other** contributes **103** customer accounts, making it the second-largest source.
- **Ads** and **Event** generate a similar number of customers, with **98** and **96** accounts respectively.
- **Partner** contributes the fewest customer accounts (**89**) among the available referral sources.

## Question 7

### Business Question

What percentage of all accounts are currently marked as churned (`churn_flag = TRUE`)?

### Business Objective

Calculate the overall customer churn rate, one of the most important SaaS Key Performance Indicators (KPIs). This metric helps measure customer retention and the percentage of customers who have stopped using the service.

 Difficulty

Beginner

SQL Concepts Required

- CASE
- SUM()
- COUNT()
- Conditional Aggregation
- ROUND()

In [83]:
query = """
select round((sum(case when churn_flag ='True' then 1 else 0 end)* 100) / count(*),2) as churn_percentage
from accounts;
"""
pd.read_sql(query, eng)

,churn_percentage
0,22.0


### Observation

- The overall customer churn rate is **22.00%**.
- This indicates that approximately **22 out of every 100 customer accounts** have churned.

## Question 8

### Business Question

What is the distribution of customer accounts across different plan tiers (Basic, Pro, Enterprise)?

### Business Objective

Understand the product mix by identifying which subscription plan has the highest number of customers. This helps evaluate product adoption and supports pricing and product strategy decisions.

Difficulty

Beginner

SQL Concepts Required

- SELECT
- COUNT()
- GROUP BY
- ORDER BY
- Subquery
- Percentage Calculation

In [84]:
query = """
select plan_tier , count(*) as no_of_accounts ,
((count(*)*100) / (select count(*) from accounts)) as percentage_calculation  from accounts 
group by plan_tier order by no_of_accounts desc;

"""
pd.read_sql(query, eng)

,plan_tier,no_of_accounts,percentage_calculation
0,Pro,178,35.6
1,Basic,168,33.6
2,Enterprise,154,30.8


### Observation

- The customer accounts are distributed across three subscription plans.
- **Pro** is the most popular plan with **178 accounts (35.60%)**.
- **Basic** is the second most popular plan with **168 accounts (33.60%)**.
- **Enterprise** has **154 accounts (30.80%)**, making it the least adopted plan.
- The distribution is fairly balanced, with each plan representing approximately one-third of the customer base.

## Question 9

### Business Question

What is the earliest and latest signup date in the `accounts` table? What is the full date range covered by the dataset?

### Business Objective

Determine the time period covered by the dataset before performing any time-based or trend analysis.

Difficulty

Beginner

SQL Concepts Required

- MIN()
- MAX()
- Date Functions

In [85]:
query = """
SELECT 
    MIN(signup_date) AS earliest_signup_date, 
    MAX(signup_date) AS latest_signup_date 
FROM accounts;

"""
pd.read_sql(query, eng)

,earliest_signup_date,latest_signup_date
0,2023-01-02,2024-12-31


### Observation

- The earliest account signup date is **2023-01-02**.
- The latest account signup date is **2024-12-31**.
- The dataset covers customer signups over a **two-year period**, making it suitable for time-based trend analysis.

## Question 10

### Business Question

For each account, count how many subscription records exist in the `subscriptions` table. Are there accounts with more than one subscription record?

### Business Objective

Understand the subscription lifecycle of customer accounts. Multiple subscription records may indicate renewals, plan upgrades, downgrades, or repeated subscription periods, which is important when calculating customer-level and subscription-level metrics.

Difficulty

Beginner – Intermediate

SQL Concepts Required

- SELECT
- LEFT JOIN
- COUNT()
- GROUP BY
- HAVING
- ORDER BY

In [86]:
query = """
select a.account_id ,a.account_name, count(s.subscription_id) as subscription_count from accounts as a 
left join subscriptions as s on 
s.account_id = a.account_id 
group by a.account_id , a.account_name
having count(s.subscription_id) > 1 order by subscription_count desc;
"""
pd.read_sql(query, eng)

,account_id,account_name,subscription_count
0,A-592832,Company_10,19
1,A-726cfa,Company_82,19
2,A-d4ac0e,Company_364,19
3,A-5a92e7,Company_413,19
4,A-8bde0c,Company_287,18
...,...,...,...
495,A-038089,Company_156,3
496,A-342303,Company_294,3
497,A-149a69,Company_302,3
498,A-f19b24,Company_462,3


# **Level 2: Intermediate (Subscription & Revenue Analysis, Time-Based Analysis, Joins)**

## 📋 RavenStack SQL Roadmap — Set 2 of 4 (Questions 11–20)

## Question 11

### Business Question

What is the total MRR (Monthly Recurring Revenue) and total ARR (Annual Recurring Revenue) currently active in the business?

### Business Objective

Calculate the core recurring revenue metrics for active subscriptions. These metrics provide a high-level view of the company's current recurring revenue base.

Difficulty

Intermediate

SQL Concepts Required

- WHERE
- SUM()
- IS NULL

In [87]:
query = """
select sum(mrr_amount) as Monthly_Recurring_Revenue , sum(arr_amount) as Annual_Recurring_Revenue from subscriptions
where end_date is null or end_date = '' ;
"""
pd.read_sql(query, eng)

,Monthly_Recurring_Revenue,Annual_Recurring_Revenue
0,10159608.0,121915296.0


### Observation

- The active subscriptions generate a total MRR of **$10,159,608**.
- 
- The  corresponding   total ARR is **$121,915,296**.
- These values represent the recurring revenue currently associated with subscriptions that have no end date.

## Question 12

### Business Question

What is the average MRR per account, broken down by plan_tier?

### Business Objective

Understand which plan tier generates the most revenue per customer and identify potential opportunities for upselling customers to higher-value plans.

 Difficulty

Intermediate

SQL Concepts Required

- AVG()
- GROUP BY
- JOIN
- ORDER BY

In [88]:
query = """
SELECT 
    s.plan_tier,
    AVG(s.mrr_amount) AS avg_mrr_amount
FROM accounts a
JOIN subscriptions s 
    ON a.account_id = s.account_id
GROUP BY s.plan_tier
ORDER BY avg_mrr_amount DESC;
"""
pd.read_sql(query, eng)

,plan_tier,avg_mrr_amount
0,Enterprise,4917.7139
1,Pro,1256.7696
2,Basic,474.6798


### Observation

- Enterprise has the highest average MRR at **$4,917.71**.
- Pro has an average MRR of **$1,256.77**.
- Basic has the lowest average MRR at **$474.68**.
- Enterprise subscriptions generate substantially higher average recurring revenue than Pro and Basic.

## Question 13

### Business Question

How many subscriptions are currently active versus churned, based on `end_date` being NULL or not NULL?

### Business Objective

Distinguish subscription-level churn from account-level churn and understand the current status of subscription records.

Difficulty

Intermediate

SQL Concepts Required

- CASE WHEN
- SUM()
- Conditional Aggregation
- IS NULL
- IS NOT NULL

In [89]:
query = """
select sum(case when end_date = '' or end_date is null  then 1 else 0 end ) as active_subscriptions ,
sum(case when end_date <> '' and end_date is not null then 1 else 0 end) as churned_subscriptions from subscriptions;

"""
pd.read_sql(query, eng)

,active_subscriptions,churned_subscriptions
0,4514.0,486.0


### Observation

- There are **4,514 active subscriptions**.
- There are **486 churned subscriptions**.
- Active subscriptions represent the majority of subscription records in the dataset.

## Question 14

### Business Question

What is the month-over-month trend of new subscriptions started, from the earliest to the latest month in the dataset?

### Business Objective

Identify subscription growth trends and potential seasonality in customer acquisition.

Difficulty

Intermediate

SQL Concepts Required

- DATE_FORMAT()
- GROUP BY
- 
- ORDER BY
- CTE

In [90]:
query = """
with cte as ( select * , date_format(start_date , '%%Y-%%m-01') as subscription_month from subscriptions) 
select subscription_month , count(*) as no_of_new_subscriptions from cte 
group by subscription_month 
order by subscription_month asc;
"""
pd.read_sql(query, eng)

,subscription_month,no_of_new_subscriptions
0,2023-01-01,3
1,2023-02-01,11
2,2023-03-01,17
3,2023-04-01,31
4,2023-05-01,29
5,2023-06-01,46
6,2023-07-01,59
7,2023-08-01,82
8,2023-09-01,62
9,2023-10-01,84


### Observation

- New subscriptions increased substantially over the period from January 2023 to December 2024.
- Subscription starts increased from **3 in January 2023** to **953 in December 2024**.
- The growth became particularly strong during the second half of 2024.
- December 2024 recorded the highest number of new subscriptions in the dataset.

##  Question 15
 Business Question: What is the trial-to-paid conversion rate?(What percentage of subscriptions that started as is_trial = TRUE are still active or converted, vs. those that churned during/after trial?)

## Business Objective:
Measure how effectively free trials convert into paying, retained customers — a critical SaaS growth metric.

Difficulty: Intermediate

SQL Concepts Required:
CASE WHEN, conditional aggregation, percentage calculation


In [91]:
query = """
SELECT 
    SUM(CASE WHEN is_trial = 'True' THEN 1 ELSE 0 END) AS total_trials,
    SUM(CASE WHEN is_trial = 'True' AND churn_flag = 'False' THEN 1 ELSE 0 END) AS converted_trials,
    ROUND(
        (SUM(CASE WHEN is_trial = 'True' AND churn_flag = 'False' THEN 1 ELSE 0 END) * 100.0) / 
        NULLIF(SUM(CASE WHEN is_trial = 'True' THEN 1 ELSE 0 END), 0), 
        2
    ) AS conversion_rate_percentage
FROM subscriptions;

"""
pd.read_sql(query, eng)

,total_trials,converted_trials,conversion_rate_percentage
0,778.0,700.0,89.97


### Observation

- There were **778 trial subscriptions** in the dataset.
- **700 trial subscriptions** were not marked as churned.
- Based on this definition, the non-churned trial rate is **89.97%**.
- However, this should not be interpreted as a confirmed trial-to-paid conversion rate because the available tables do not contain an explicit field identifying whether a trial converted to a paid subscription.

## Question 16

### Business Question

How many subscriptions had an `upgrade_flag = TRUE` versus a `downgrade_flag = TRUE`? What is the upgrade-to-downgrade ratio?

### Business Objective

Understand product expansion versus contraction behavior within the existing customer base.

 Difficulty

Intermediate

 SQL Concepts Required

- CASE WHEN
- Conditional Aggregation
- SUM()
- Ratio Calculation
- NULLIF()

In [92]:
query = """
SELECT 
    SUM(CASE WHEN upgrade_flag = 'True' THEN 1 ELSE 0 END) AS total_upgrades,
    SUM(CASE WHEN downgrade_flag = 'True' THEN 1 ELSE 0 END) AS total_downgrades,
    ROUND(
        SUM(CASE WHEN upgrade_flag = 'True' THEN 1 ELSE 0 END) / 
        NULLIF(SUM(CASE WHEN downgrade_flag = 'True' THEN 1 ELSE 0 END), 0), 
        2
    ) AS net_upgrade_ratio
FROM subscriptions;
"""
pd.read_sql(query, eng)

,total_upgrades,total_downgrades,net_upgrade_ratio
0,529.0,218.0,2.43


### Observation

- There were **529 upgrades** and **218 downgrades**.
- The upgrade-to-downgrade ratio is **2.43**.
- This means there were approximately **2.43 upgrades for every 1 downgrade**.

## Question 17

### Business Question

What is the total MRR broken down by billing frequency (monthly vs. annual)? What percentage of MRR comes from each billing frequency?

### Business Objective

Understand the billing preference mix and the contribution of monthly and annual billing to total MRR.

Difficulty

Intermediate

 SQL Concepts Required

- GROUP BY
- SUM()
- Subquery
- Percentage Calculation

In [93]:
query = """
SELECT 
    billing_frequency,
    SUM(mrr_amount) AS total_mrr,
    ROUND(
        (SUM(mrr_amount) * 100.0) / (SELECT SUM(mrr_amount) FROM subscriptions), 
        2
    ) AS revenue_percentage
FROM subscriptions
GROUP BY billing_frequency;
"""
pd.read_sql(query, eng)

,billing_frequency,total_mrr,revenue_percentage
0,monthly,5741349.0,50.63
1,annual,5597398.0,49.37


### Observation

- Monthly billing contributes **5.74M (50.63%)** of total MRR.
- Annual billing contributes **5.60M (49.37%)**.
- The MRR contribution is almost evenly split between monthly and annual billing.

## Question 18

### Business Question

Which industry generates the highest total MRR? Join the `accounts` and `subscriptions` tables to find total active revenue per industry.

### Business Objective

Identify the most valuable customer segment based on active recurring revenue, rather than only the number of customer accounts.

Difficulty

Intermediate – Advanced

SQL Concepts Required

- INNER JOIN
- GROUP BY
- SUM()
- WHERE
- ORDER BY

In [94]:
query = """
SELECT 
    a.industry, 
    SUM(s.mrr_amount) AS total_mrr 
FROM accounts AS a
JOIN subscriptions AS s 
    ON a.account_id = s.account_id
WHERE s.end_date = '' OR s.end_date IS NULL
GROUP BY a.industry
ORDER BY total_mrr DESC;
"""
pd.read_sql(query, eng)

,industry,total_mrr
0,FinTech,2416050.0
1,DevTools,2165043.0
2,Cybersecurity,1974256.0
3,EdTech,1844424.0
4,HealthTech,1759835.0


### Observation

- **FinTech** generates the highest active MRR at **2,416,050**.
- **DevTools** is the second-highest revenue-generating industry with **2,165,043**.
- **HealthTech** generates the lowest active MRR at **1,759,835**.
- The ranking by revenue differs from the account-count ranking in Question 5, showing that customer volume does not necessarily equal revenue contribution.

## Question 19

### Business Question

What is the churn rate among Enterprise plan customers compared with Basic and Pro? Does a higher-tier plan appear to have a lower churn rate?

### Business Objective

Compare subscription-level churn across plan tiers and evaluate whether higher-value plans show lower churn.

Difficulty

Intermediate – Advanced

SQL Concepts Required

- GROUP BY
- CASE WHEN
- COUNT()
- Conditional Aggregation
- Percentage Calculation
- ORDER BY

In [95]:
query = """
SELECT 
    plan_tier,
    COUNT(*) AS total_subscriptions,
    SUM(CASE WHEN churn_flag = 'True' THEN 1 ELSE 0 END) AS churned_subscriptions,
    ROUND(
        (SUM(CASE WHEN churn_flag = 'True' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 
        2
    ) AS tier_churn_rate_percentage
FROM subscriptions
GROUP BY plan_tier
ORDER BY tier_churn_rate_percentage DESC;

"""
pd.read_sql(query, eng)

,plan_tier,total_subscriptions,churned_subscriptions,tier_churn_rate_percentage
0,Enterprise,1723,172.0,9.98
1,Pro,1675,162.0,9.67
2,Basic,1602,152.0,9.49


### Observation

- Enterprise has the highest subscription churn rate at **9.98%**.
- Pro has a churn rate of **9.67%**.
- Basic has the lowest churn rate at **9.49%**.
- The difference between the highest and lowest churn rates is only **0.49 percentage points**, so the churn rates are relatively similar across plan tiers.

## Question 20

### Business Question

For accounts acquired through each referral source, what is the average MRR per account? Which acquisition channel brings the highest-value customers?

### Business Objective

Connect customer acquisition channels with revenue outcomes to identify which channels attract higher-value customers.

Difficulty

Advanced

SQL Concepts Required

- JOIN
- GROUP BY
- AVG()
- ORDER BY

In [96]:
query = """
select a.referral_source , avg(s.mrr_amount) as avg_mrr_amount from accounts as a
join subscriptions as s on 
a.account_id = s.account_id 
group by a.referral_source order by avg_mrr_amount desc;
"""
pd.read_sql(query, eng)

,referral_source,avg_mrr_amount
0,organic,2392.0573
1,partner,2363.7054
2,other,2315.2171
3,ads,2146.6190
4,event,2095.3653


### Observation

- **Organic** has the highest average MRR at **2,392.06**.
- **Partner** is the second-highest at **2,363.71**.
- **Event** has the lowest average MRR at **2,095.37**.
- The difference between the highest and lowest average MRR is relatively small.
- The result represents average MRR per subscription record, not strictly per unique account.
  

# **📋 RavenStack SQL Roadmap — Set 3 of 3 (Questions 21–30)**

## Level 3: Advanced (Window Functions, CTEs, Subqueries, Ranking, Segmentation, Cohorts)

## Question 21

### Business Question

Rank all accounts by their total MRR within each industry. Identify the top revenue-generating accounts in each industry.

### Business Objective

Identify the highest-value accounts within each industry segment to support account management prioritization and customer success strategies.

Difficulty

Advanced

SQL Concepts Required

- INNER JOIN
- GROUP BY
- SUM()
- CTE
- DENSE_RANK()
- PARTITION BY
- ORDER BY

In [97]:
query = """
WITH account_revenue AS (
    SELECT 
        a.account_id,
        a.account_name,
        a.industry,
        SUM(s.mrr_amount) AS total_mrr
    FROM accounts AS a
    JOIN subscriptions AS s 
        ON a.account_id = s.account_id
    GROUP BY 
        a.account_id,
        a.account_name,
        a.industry
),
ranked_accounts AS (
    SELECT 
        *,
        DENSE_RANK() OVER (
            PARTITION BY industry 
            ORDER BY total_mrr DESC
        ) AS revenue_rank
    FROM account_revenue
)
SELECT *
FROM ranked_accounts
WHERE revenue_rank <= 5
ORDER BY industry, revenue_rank;
"""

pd.read_sql(query, eng)

,account_id,account_name,industry,total_mrr,revenue_rank
0,A-4814a3,Company_337,Cybersecurity,87957.0,1
1,A-80eeb6,Company_402,Cybersecurity,68489.0,2
2,A-9174e0,Company_73,Cybersecurity,53976.0,3
3,A-118f1c,Company_492,Cybersecurity,52160.0,4
4,A-7920cc,Company_412,Cybersecurity,52089.0,5
5,A-5b1bcd,Company_166,DevTools,133298.0,1
6,A-afa505,Company_157,DevTools,66010.0,2
7,A-aa9511,Company_203,DevTools,60970.0,3
8,A-bd4513,Company_205,DevTools,52204.0,4
9,A-d77f4c,Company_470,DevTools,48726.0,5


### Observation

- The query identified the **top 5 revenue-generating accounts within each industry**.
- The highest-ranked account overall is **Company_166 in DevTools**, with total MRR of **$133,298**.
- FinTech's top account, **Company_403**, generates **$138,060**, the highest individual account MRR in the results.
- Each industry has five ranked accounts, from rank 1 to rank 5.
- The results show that revenue contribution varies significantly among accounts within the same industry.

## Question 22

### Business Question

For each plan tier, find the top 3 highest-paying accounts based on total MRR.

### Business Objective

Identify high-value customers within each plan tier for account management, customer success, and potential case-study opportunities.

Difficulty

Advanced

SQL Concepts Required

- INNER JOIN
- GROUP BY
- SUM()
- ROW_NUMBER()
- PARTITION BY
- CTE
- ORDER BY

In [98]:
query = """
WITH vip AS (
    SELECT 
        a.account_id,
        a.account_name,
        s.plan_tier,
        SUM(s.mrr_amount) AS total_mrr,
        ROW_NUMBER() OVER (
            PARTITION BY s.plan_tier
            ORDER BY SUM(s.mrr_amount) DESC
        ) AS revenue_rank
    FROM accounts AS a
    JOIN subscriptions AS s
        ON a.account_id = s.account_id
    GROUP BY 
        a.account_id,
        a.account_name,
        s.plan_tier
)
SELECT *
FROM vip
WHERE revenue_rank <= 3
ORDER BY plan_tier, revenue_rank;
"""

pd.read_sql(query, eng)

,account_id,account_name,plan_tier,total_mrr,revenue_rank
0,A-4a4c2d,Company_473,Basic,7182.0,1
1,A-fa3095,Company_174,Basic,6650.0,2
2,A-2d1036,Company_399,Basic,6555.0,3
3,A-d4e0d4,Company_403,Enterprise,116415.0,1
4,A-5b1bcd,Company_166,Enterprise,116216.0,2
5,A-5a215a,Company_358,Enterprise,103878.0,3
6,A-18793f,Company_488,Pro,43512.0,1
7,A-40906c,Company_235,Pro,22540.0,2
8,A-5a215a,Company_358,Pro,21315.0,3


### Observation

- The query identified the top 3 highest-MRR accounts within each plan tier.
- The **Enterprise** tier has the highest-value accounts, with the top account generating **116,415** in total MRR.
- The top **Basic** account generates **7,182** in total MRR.
- The top **Pro** account generates **43,512** in total MRR.
- Enterprise accounts in the top 3 contribute substantially more MRR than the top accounts in Basic and Pro.

## Question 23

### Business Question

Calculate the cumulative (running total) MRR added by new subscriptions month over month, from the start of the dataset to the end.

### Business Objective

Measure the cumulative growth in MRR from new subscription starts and understand the overall revenue growth trajectory over time.

Difficulty

Advanced

SQL Concepts Required

- CTE
- DATE_FORMAT()
- SUM()
- Window Function
- SUM() OVER()
- ORDER BY

In [99]:
query = """
WITH monthly_revenue AS (
    SELECT 
        DATE_FORMAT(start_date, '%%Y-%%m-01') AS subscription_month,
        SUM(mrr_amount) AS monthly_mrr
    FROM subscriptions
    GROUP BY subscription_month
)
SELECT 
    subscription_month,
    monthly_mrr,
    SUM(monthly_mrr) OVER (
        ORDER BY subscription_month ASC
    ) AS cumulative_mrr
FROM monthly_revenue
ORDER BY subscription_month;
"""

pd.read_sql(query, eng)

,subscription_month,monthly_mrr,cumulative_mrr
0,2023-01-01,4684.0,4684.0
1,2023-02-01,11079.0,15763.0
2,2023-03-01,25885.0,41648.0
3,2023-04-01,41788.0,83436.0
4,2023-05-01,85919.0,169355.0
5,2023-06-01,74987.0,244342.0
6,2023-07-01,120422.0,364764.0
7,2023-08-01,165621.0,530385.0
8,2023-09-01,116222.0,646607.0
9,2023-10-01,182518.0,829125.0


### Observation

- Monthly MRR from new subscriptions generally increased throughout the dataset period, although some months experienced temporary declines.
- Monthly MRR increased from **4,684 in January 2023** to **2,273,427 in December 2024**.
- The cumulative MRR increased continuously from **4,684** to **11,338,747** over the period.
- The strongest monthly MRR was recorded in **December 2024**, with **2,273,427**.
- The growth became particularly strong during the second half of 2024.

## Question 24

### Business Question

What is the month-over-month percentage growth rate in new subscriptions by comparing each month's subscription count with the previous month?

### Business Objective

Measure subscription growth momentum rather than only looking at raw subscription counts. This helps identify periods of accelerating or slowing customer acquisition.

Difficulty

Advanced

SQL Concepts Required

- CTE
- COUNT()
- LAG()
- Window Function
- Percentage Growth Calculation
- NULLIF()
- ORDER BY

In [100]:
query = """
WITH monthly_subs AS (
    SELECT 
        DATE_FORMAT(start_date, '%%Y-%%m-01') AS subscription_month,
        COUNT(*) AS new_subscriptions
    FROM subscriptions
    GROUP BY subscription_month
),
monthly_lag AS (
    SELECT 
        subscription_month,
        new_subscriptions,
        LAG(new_subscriptions) OVER (
            ORDER BY subscription_month
        ) AS previous_month_subs
    FROM monthly_subs
)
SELECT 
    subscription_month,
    new_subscriptions,
    previous_month_subs,
    ROUND(
        (new_subscriptions - previous_month_subs) * 100.0 /
        NULLIF(previous_month_subs, 0),
        2
    ) AS mom_growth_percentage
FROM monthly_lag
ORDER BY subscription_month;
"""

pd.read_sql(query, eng)

,subscription_month,new_subscriptions,previous_month_subs,mom_growth_percentage
0,2023-01-01,3,NaN,NaN
1,2023-02-01,11,3.0,266.67
2,2023-03-01,17,11.0,54.55
3,2023-04-01,31,17.0,82.35
4,2023-05-01,29,31.0,-6.45
5,2023-06-01,46,29.0,58.62
6,2023-07-01,59,46.0,28.26
7,2023-08-01,82,59.0,38.98
8,2023-09-01,62,82.0,-24.39
9,2023-10-01,84,62.0,35.48


### Observation

- January 2023 has no previous month, so its MoM growth is not calculated.
- New subscriptions generally increased over the dataset period, although some months experienced declines.
- The largest decline occurred in **September 2023**, when new subscriptions decreased by **24.39%** compared with August.
- The strongest growth occurred in **February 2023**, with a **266.67%** increase from January.
- In 2024, subscription growth was mostly positive, with the largest increase occurring in **December 2024 (55.72%)**.
- New subscriptions increased from **3 in January 2023** to **953 in December 2024**.

## Question 25

### Business Question

Segment all accounts into "High Value," "Medium Value," and "Low Value" tiers based on their total MRR. How many accounts fall into each segment?

### Business Objective

Create a simple customer value segmentation to identify high-value, medium-value, and low-value accounts for targeted retention and upsell strategies.

Difficulty

Advanced

SQL Concepts Required

- CTE
- CASE WHEN
- SUM()
- GROUP BY
- Subquery
- Customer Segmentation

In [101]:
query = """
WITH account_mrr AS (
    SELECT 
        a.account_id,
        a.account_name,
        SUM(s.mrr_amount) AS total_mrr
    FROM accounts AS a
    JOIN subscriptions AS s 
        ON a.account_id = s.account_id
    GROUP BY 
        a.account_id,
        a.account_name
),
segmented_accounts AS (
    SELECT 
        account_id,
        account_name,
        total_mrr,
        CASE 
            WHEN total_mrr >= 25000 THEN 'High Value'
            WHEN total_mrr >= 10000 THEN 'Medium Value'
            ELSE 'Low Value'
        END AS value_segment
    FROM account_mrr
)
SELECT 
    value_segment,
    COUNT(*) AS account_count
FROM segmented_accounts
GROUP BY value_segment
ORDER BY account_count DESC;
"""

pd.read_sql(query, eng)

,value_segment,account_count
0,Medium Value,244
1,High Value,165
2,Low Value,91


### Observation

- **Medium Value** accounts are the largest segment with **244 accounts**.
- **High Value** accounts include **165 accounts**.
- **Low Value** accounts include **91 accounts**.
- Most accounts fall into the **Medium Value** segment based on the selected MRR thresholds.

## Question 26

### Business Question

Group accounts into signup cohorts by month. What is the churn rate for each signup cohort?

### Business Objective

Perform cohort-level churn analysis to determine whether customers who signed up in different periods show different churn patterns.

Difficulty

Advanced

SQL Concepts Required

- CTE
- DATE_FORMAT()
- GROUP BY
- CASE WHEN
- Conditional Aggregation
- Percentage Calculation

In [102]:
query = """
WITH account_cohorts AS (
    SELECT 
        a.account_id,
        DATE_FORMAT(a.signup_date, '%%Y-%%m-01') AS signup_cohort,
        a.churn_flag
    FROM accounts AS a
),
cohort_summary AS (
    SELECT 
        signup_cohort,
        COUNT(*) AS total_accounts_in_cohort,
        SUM(CASE WHEN churn_flag = 'True' THEN 1 ELSE 0 END) AS churned_accounts
    FROM account_cohorts
    GROUP BY signup_cohort
)
SELECT 
    signup_cohort,
    total_accounts_in_cohort,
    churned_accounts,
    ROUND(
        churned_accounts * 100.0 / NULLIF(total_accounts_in_cohort, 0),
        2
    ) AS cohort_churn_rate_percentage
FROM cohort_summary
ORDER BY signup_cohort;
"""

pd.read_sql(query, eng)

,signup_cohort,total_accounts_in_cohort,churned_accounts,cohort_churn_rate_percentage
0,2023-01-01,17,4.0,23.53
1,2023-02-01,18,7.0,38.89
2,2023-03-01,20,5.0,25.00
3,2023-04-01,15,3.0,20.00
4,2023-05-01,26,7.0,26.92
5,2023-06-01,13,3.0,23.08
6,2023-07-01,14,3.0,21.43
7,2023-08-01,16,3.0,18.75
8,2023-09-01,23,5.0,21.74
9,2023-10-01,20,7.0,35.00


### Observation

- Churn rates vary considerably across signup cohorts.
- The **February 2023 cohort** has the highest churn rate at **38.89%**.
- The **September 2024 cohort** has a churn rate of **36.00%**, the second-highest among the cohorts shown.
- The **August 2024 cohort** has the lowest churn rate at **9.52%** among the larger recent cohorts.
- Recent cohorts such as October, November, and December 2024 show relatively lower churn rates, but these cohorts may have had less time to experience churn.

## Question 27

### Business Question

Which accounts have an average subscription MRR higher than the overall average MRR across all subscriptions?

### Business Objective

Identify above-average-value customers without using a hardcoded revenue threshold. This helps identify customers whose average subscription value is higher than the overall subscription benchmark.

### Difficulty

Advanced

### SQL Concepts Required

- CTE
- AVG()
- GROUP BY
- Subquery
- CROSS JOIN
- WHERE
- ORDER BY

In [103]:
query = """
WITH account_avg_mrr AS (
    SELECT 
        a.account_id,
        a.account_name,
        AVG(s.mrr_amount) AS avg_account_mrr
    FROM accounts AS a
    JOIN subscriptions AS s 
        ON a.account_id = s.account_id
    GROUP BY 
        a.account_id,
        a.account_name
),
global_avg AS (
    SELECT 
        AVG(mrr_amount) AS overall_avg_mrr
    FROM subscriptions
)
SELECT 
    t.account_id,
    t.account_name,
    t.avg_account_mrr,
    g.overall_avg_mrr
FROM account_avg_mrr AS t
CROSS JOIN global_avg AS g
WHERE t.avg_account_mrr > g.overall_avg_mrr
ORDER BY t.avg_account_mrr DESC;
"""

pd.read_sql(query, eng)

,account_id,account_name,avg_account_mrr,overall_avg_mrr
0,A-d4e0d4,Company_403,13806.0000,2267.7494
1,A-5c046d,Company_130,10485.7500,2267.7494
2,A-c58f49,Company_480,10140.0000,2267.7494
3,A-30b4ca,Company_23,9492.2500,2267.7494
4,A-5b1bcd,Company_166,8886.5333,2267.7494
...,...,...,...,...
182,A-6965e1,Company_379,2313.7500,2267.7494
183,A-7dacce,Company_8,2305.7000,2267.7494
184,A-812c5b,Company_395,2298.7692,2267.7494
185,A-410e86,Company_210,2280.0000,2267.7494


### Observation

- The overall average MRR across all subscriptions is **2,267.75**.
- **187 accounts** have an average subscription MRR above this overall average.
- Company_403 has the highest average account MRR at **13,806.00**.
- Company_130 and Company_480 also have significantly higher average MRR than the overall benchmark.
- The results identify accounts whose average subscription value is above the overall subscription-level average.

## Question 28

### Business Question

For accounts with more than one subscription record, calculate their total lifetime MRR sum across all subscription records and rank them from highest to lowest.

### Business Objective

Identify high-value customers with multiple subscription records and prioritize them for loyalty, retention, and customer success strategies.

### Difficulty

Advanced

### SQL Concepts Required

- CTE
- COUNT()
- HAVING
- SUM()
- RANK()
- Window Function
- ORDER BY

In [104]:
query = """
WITH multi_sub_accounts AS (
    SELECT 
        a.account_id,
        a.account_name,
        COUNT(s.subscription_id) AS total_subscriptions,
        SUM(s.mrr_amount) AS lifetime_mrr_sum
    FROM accounts AS a
    JOIN subscriptions AS s 
        ON a.account_id = s.account_id
    GROUP BY 
        a.account_id,
        a.account_name
    HAVING COUNT(s.subscription_id) > 1
),
ranked_lifetime_value AS (
    SELECT 
        account_id,
        account_name,
        total_subscriptions,
        lifetime_mrr_sum,
        RANK() OVER (
            ORDER BY lifetime_mrr_sum DESC
        ) AS ltv_rank
    FROM multi_sub_accounts
)
SELECT *
FROM ranked_lifetime_value
ORDER BY ltv_rank ASC;
"""

pd.read_sql(query, eng)

,account_id,account_name,total_subscriptions,lifetime_mrr_sum,ltv_rank
0,A-d4e0d4,Company_403,10,138060.0,1
1,A-5b1bcd,Company_166,15,133298.0,2
2,A-5a215a,Company_358,18,130152.0,3
3,A-1f0636,Company_368,11,94710.0,4
4,A-4814a3,Company_337,11,87957.0,5
...,...,...,...,...,...
495,A-bf7919,Company_426,6,1560.0,496
496,A-0be015,Company_454,5,1178.0,497
497,A-44dc83,Company_35,4,456.0,498
498,A-1e6fc3,Company_434,4,444.0,499


### Observation

- The analysis identified **500 accounts** with more than one subscription record.
- Company_403 has the highest lifetime MRR sum at **138,060**, with 10 subscription records.
- Company_166 ranks second with **133,298** across 15 subscription records.
- Company_358 ranks third with **130,152** across 18 subscription records.
- The lifetime MRR sum varies significantly across accounts, with the highest-ranked accounts contributing substantially more recurring revenue across their subscription records.

## Question 29

### Business Question

What is the average customer tenure, in days, from signup date to the end of their subscription lifecycle for churned accounts, or to today for currently active accounts?

### Business Objective

Understand how long customers typically remain with RavenStack before churning. This can help identify potential early-warning points for customer retention strategies.

### Difficulty

Advanced

### SQL Concepts Required

- DATEDIFF()
- CASE WHEN
- AVG()
- MAX()
- GROUP BY
- JOIN
- Conditional Aggregation

### Data Limitation

The selected `accounts` and `subscriptions` tables do not contain an explicit customer-level `churn_date`. Therefore, for churned accounts, the latest `end_date` from their subscription records is used as a proxy for the end of their customer lifecycle.

In [105]:
query = """
WITH account_lifecycle AS (
    SELECT
        a.account_id,
        a.churn_flag,
        a.signup_date,
        MAX(s.end_date) AS last_subscription_end_date
    FROM accounts AS a
    LEFT JOIN subscriptions AS s
        ON a.account_id = s.account_id
    GROUP BY
        a.account_id,
        a.churn_flag,
        a.signup_date
)
SELECT
    churn_flag,
    COUNT(*) AS account_count,
    ROUND(
        AVG(
            DATEDIFF(
                CASE
                    WHEN churn_flag = 'True'
                        THEN last_subscription_end_date
                    ELSE CURDATE()
                END,
                signup_date
            )
        ),
        2
    ) AS average_tenure_days
FROM account_lifecycle
GROUP BY churn_flag
ORDER BY churn_flag DESC;
"""

pd.read_sql(query, eng)

,churn_flag,account_count,average_tenure_days
0,True,110,294.97
1,False,390,919.43


### Observation

- There are **110 churned accounts**, with an average tenure of **294.97 days**.
- There are **390 active accounts**, with an average tenure of **919.43 days**.
- Active accounts have a substantially longer average tenure than churned accounts.
- The average tenure difference is approximately **624 days**.

## Question 30

### Business Question

Build a single Executive KPI Summary query that returns in one row:

- Total accounts
- Active subscriptions
- Total active MRR
- Overall churn rate %
- Average active MRR per account

### Business Objective

Simulate a real executive dashboard request by combining the key SaaS business KPIs into a single summary row for quick decision-making.

### Difficulty

Advanced

### SQL Concepts Required

- Multiple subqueries
- SELECT
- COUNT()
- SUM()
- AVG()
- Conditional Aggregation
- DISTINCT
- Percentage Calculation

In [106]:
query = """
SELECT 
    (SELECT COUNT(*) 
     FROM accounts) AS total_accounts,

    (SELECT COUNT(*) 
     FROM subscriptions 
     WHERE end_date = '' OR end_date IS NULL) AS active_subscriptions,

    (SELECT SUM(mrr_amount) 
     FROM subscriptions 
     WHERE end_date = '' OR end_date IS NULL) AS total_active_mrr,

    (SELECT ROUND(
        SUM(CASE WHEN churn_flag = 'True' THEN 1 ELSE 0 END) * 100.0 
        / COUNT(*),
        2
    )
     FROM accounts) AS overall_churn_rate_pct,

    (SELECT ROUND(
        SUM(mrr_amount) / COUNT(DISTINCT account_id),
        2
    )
     FROM subscriptions
     WHERE end_date = '' OR end_date IS NULL) AS avg_active_mrr_per_account;
"""

pd.read_sql(query, eng)

,total_accounts,active_subscriptions,total_active_mrr,overall_churn_rate_pct,avg_active_mrr_per_account
0,500,4514,10159608.0,22.0,20319.22


### Observation

- RavenStack has **500 total accounts**.
- There are **4,514 active subscriptions**.
- Total active MRR is **10,159,608**.
- The overall account churn rate is **22.00%**.
- Average active MRR per account is **20,319.22**.
- The KPI query successfully combines the major business metrics into a single row.

# Project Conclusion

This analysis covered **500 accounts** and **5,000 subscription records** from the RavenStack Synthetic SaaS dataset. Using SQL, I performed data validation and answered **30 business questions** covering customer, subscription, revenue, acquisition, segmentation, and churn analytics.

## Key Findings

- Overall account churn rate is **22%**, with churned accounts averaging approximately **295 days** of tenure compared with **919 days** for active accounts, indicating a potential early-stage retention risk.
- **DevTools** is the leading industry by account count and total revenue, while **Enterprise** customers generate the highest average MRR per customer.
- Total active MRR is approximately **$10.16M**, with **Organic** acquisition showing the highest average customer value among the referral sources analyzed.
- Customer value is unevenly distributed, with **165 High Value**, **244 Medium Value**, and **91 Low Value** accounts, providing a basis for customer prioritization and retention analysis.

## Next Step

The SQL analysis is now complete. The next phase is to use these findings to build an interactive **Power BI dashboard** for monitoring revenue, subscriptions, customers, acquisition, and churn-related KPIs.

---

*Dataset: RavenStack Synthetic SaaS Dataset by River @ Rivalytics*